In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
def Attention(Q, K, V, mask=None):

    d_k = Q.shape[-1]
    adj_matrix = Q @ K.T / math.sqrt(d_k)

    if mask is not None:

        n = adj_matrix.shape[0]

        for k in range(n):
            for j in range(n):
                if j > k:
                    adj_matrix[k][j] = -float("inf")

    attention = torch.softmax(adj_matrix, dim=-1) @ V

    return attention

In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(self):

    super().__init__()
    self.W_O = nn.Parameter(torch.zeros(512, 512))
    self.W_Q = nn.ParameterList()
    self.W_K = nn.ParameterList()
    self.W_V = nn.ParameterList()

    for i in range(8):

      self.W_Q.append(nn.Parameter(torch.zeros(512, 64)))
      self.W_K.append(nn.Parameter(torch.zeros(512, 64)))
      self.W_V.append(nn.Parameter(torch.zeros(512, 64)))


  def forward(self,Q,K,V,mask=None):

    head = []

    for i in range(8):

      head = head + [Attention(Q @ self.W_Q[i],K @ self.W_K[i], V @ self.W_V[i])]


    head = torch.cat(head,dim=-1)
    multihead = head @ self.W_O

    return multihead

In [ ]:
class FFN(nn.Module):

  def __init__(self):

    super().__init__()
    self.linear1 = nn.Linear(512,2048)
    self.linear2 = nn.Linear(2048,512)

  def forward(self,x):

    y = self.linear1(x)
    z = nn.ReLU()(y)
    output = self.linear2(z)

    return output

In [ ]:
class Embedding(nn.Module):

  def __init__(self,n):

    super().__init__()
    self.embedding = nn.Embedding(n,512)

  def forward(self,token):

    output = self.embedding(token)

    return output

In [ ]:
def PE(EmbeddingResult):

  P = []
  n = EmbeddingResult.shape[0]

  for j in range(n):

    P_j = []

    for i in range(256):

      P_j.append(math.sin(j/10000 ** (2*i/512)))
      P_j.append(math.cos(j/10000 ** (2*i/512)))

    P.append(P_j)

  P = torch.tensor(P)

  return EmbeddingResult + P

In [ ]:
class EncoderLayer(nn.Module):

  def __init__(self):

    super().__init__()

    self.dropout = nn.Dropout(0.1)
    self.MHA = MultiHeadAttention()
    self.ffn = FFN()
    self.norm1 = nn.LayerNorm(512)
    self.norm2 = nn.LayerNorm(512)

  def forward(self,Embedding_PE_Results):

    MHA = self.MHA(Embedding_PE_Results,Embedding_PE_Results,Embedding_PE_Results)
    x1 = self.norm1(Embedding_PE_Results + self.dropout(MHA))
    x2 = self.ffn(x1)
    output = self.norm2(x1 + self.dropout(x2))

    return output



class Encoder(nn.Module):

  def __init__(self,n):

    super().__init__()
    self.dropout = nn.Dropout(0.1)
    self.embedding = Embedding(n)

    self.layer1 = EncoderLayer()
    self.layer2 = EncoderLayer()
    self.layer3 = EncoderLayer()
    self.layer4 = EncoderLayer()
    self.layer5 = EncoderLayer()
    self.layer6 = EncoderLayer()

    self.layers = nn.ModuleList([self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6])


  def forward(self,tokens):

    x = self.embedding(tokens) * math.sqrt(512)
    x = PE(x)
    x = self.dropout(x)

    for layer in self.layers:

      x = layer(x)

    return x



In [ ]:
class DecoderLayer(nn.Module):


  def __init__(self):

    super().__init__()

    self.dropout = nn.Dropout(0.1)
    self.self_MHA = MultiHeadAttention()
    self.cross_MHA = MultiHeadAttention()
    self.ffn = FFN()
    self.norm1 = nn.LayerNorm(512)
    self.norm2 = nn.LayerNorm(512)
    self.norm3 = nn.LayerNorm(512)


  def forward(self,Embedding_PE_Result,Encoder_Result):

    x = self.self_MHA(Embedding_PE_Result,Embedding_PE_Result,Embedding_PE_Result,mask=True)
    x = self.norm1(self.dropout(x)+Embedding_PE_Result)
    y = self.cross_MHA(x,Encoder_Result,Encoder_Result)
    x = self.norm2(x+self.dropout(y))
    z = self.ffn(x)
    x = self.norm3(x+self.dropout(z))

    return x


class Decoder(nn.Module):

  def __init__(self,n):

    super().__init__()
    self.dropout = nn.Dropout(0.1)
    self.embedding = Embedding(n)
    self.layer1 = DecoderLayer()
    self.layer2 = DecoderLayer()
    self.layer3 = DecoderLayer()
    self.layer4 = DecoderLayer()
    self.layer5 = DecoderLayer()
    self.layer6 = DecoderLayer()

    self.layers = nn.ModuleList([self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6])

  def forward(self,tokens,encoder_results):

      x = self.embedding(tokens) * math.sqrt(512)
      x = PE(x)
      x = self.dropout(x)

      for layer in self.layers:

        x = layer(x,encoder_results)

      return x

***Main***

In [ ]:
src_tokens = torch.tensor([12, 53, 7, 91])
tgt_tokens = torch.tensor([1, 25, 66])

encoder = Encoder(10000)
decoder = Decoder(10000)

encoder_result = encoder(src_tokens)
decoder_result = decoder(tgt_tokens, encoder_result)

print(encoder_result.shape)
print(decoder_result.shape)